In [ ]:
import tensorflow as tf

# List available GPUs
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"✅ GPU is available: {gpus}")
else:
    print("❌ No GPU found. Running on CPU.")


import pandas as pd

# Load the Excel dataset
file_path = "Bengali_Banglish_Dataset12.xlsx"  # Make sure this file is in the same directory
df = pd.read_excel(file_path)

# Display basic info
print("✅ Dataset loaded successfully!")
print("Shape of dataset:", df.shape)
print("First 5 rows:\n", df.head())

# Check for missing values
print("\nMissing values in each column:\n", df.isnull().sum())


✅ GPU is available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ Dataset loaded successfully!
Shape of dataset: (80098, 4)
First 5 rows:
   Label                                            Bengali  \
0  fear  বাংলাদেশের সীমান্ত লাগোয়া পূর্ব ভারতীয় রাজ্যটি...   
1  fear   শেরেবাংলা উচ্চবিদ্যালয়ের মাঠে গভীর নলকূপের পা...   
2  fear  তখনি হঠাৎ মিথির দিকে ভালো করে তাকাতে দেখলাম মি...   
3  fear  সন্ধ্যায় বাড়ির গৃহিণী ঘরের পাশেই কাজ করছিলেন র...   
4  fear  তাহলে কী আমাকে নিয়ে বাজে কোন ফিসফাস চালু হয়ে গ...   

                                            Banglish  \
0  bangladesher simanto lagoya purbo vartiy rajjt...   
1  sherebangla uccbidjalyer mathe govir nolkuper ...   
2  tokhni hothat mithir dike valo kore takate dek...   
3  sondhjay barir grihini ghorer pashei kaj korch...   
4  tahole ki amake niye baje kon fisfas calu hoye...   

                                             English  
0   Muslims are under severe stress in the easter...  
1  On Thursday, ga

In [ ]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# ---------------------------
# 1. Dataset already loaded as df
# ---------------------------

# ✅ Clean column names
df.columns = df.columns.str.replace("'", "").str.strip()
print("✅ Columns:", df.columns)

# ---------------------------
# 2. Select Single Column to Train On
# ---------------------------

# ✅ Choose one of these: 'Bengali', 'Banglish', 'English'
selected_column = 'English'  # Change this to 'Bengali' or 'Banglish' to test those!

# ✅ Filter dataset to selected column and label
df_single = df[[selected_column, 'Label']].dropna()
df_single.columns = ['text', 'label']  # Rename for consistency

print("\n✅ Sample Data:\n", df_single.head())

# ---------------------------
# 3. Label Encoding
# ---------------------------

label_encoder = LabelEncoder()
df_single['label'] = label_encoder.fit_transform(df_single['label'])
num_classes = len(label_encoder.classes_)

print("\n✅ Labels Encoded. Classes:", label_encoder.classes_)

# ---------------------------
# 4. Train-Test Split
# ---------------------------

X_train, X_test, y_train, y_test = train_test_split(
    df_single['text'], df_single['label'],
    test_size=0.2,
    stratify=df_single['label'],
    random_state=42
)

# ---------------------------
# 5. Tokenization and Padding
# ---------------------------

max_words = 10000  # Vocabulary size
max_len = 100      # Max sequence length

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# ---------------------------
# 6. One-hot encode labels
# ---------------------------

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

# ---------------------------
# 7. Build CNN/LSTM/GRU Model
# ---------------------------

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    # Change this line to test different models:
    # tf.keras.layers.Conv1D(128, 5, activation='relu'),  # For CNN
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=False)),  # For LSTM
    # tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64, return_sequences=False)),  # For GRU
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# ---------------------------
# 8. Train the Model
# ---------------------------

history = model.fit(
    X_train_pad, y_train_cat,
    epochs=10,
    batch_size=32,
    validation_data=(X_test_pad, y_test_cat)
)

# ---------------------------
# 9. Evaluate Model
# ---------------------------

loss, accuracy = model.evaluate(X_test_pad, y_test_cat)
print(f"\n✅ Test Accuracy on {selected_column}: {accuracy * 100:.2f}%")

# ---------------------------
# 10. Optional: Save Model
# ---------------------------

# model.save(f"text_model_{selected_column}.h5")
# print(f"\n✅ Model saved as text_model_{selected_column}.h5")


✅ Columns: Index(['Label', 'Bengali', 'Banglish', 'English'], dtype='object')

✅ Sample Data:
                                                 text label
0   Muslims are under severe stress in the easter...  fear
1  On Thursday, gas continued to escape at a high...  fear
2   Then suddenly I looked closely at Mithi and s...  fear
3  In the evening, the housewife was working next...  fear
4   So what's wrong with me has started a whisper...  fear

✅ Labels Encoded. Classes: ['anger' 'disgust' 'fear' 'joy' 'sadness' 'surprise']
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 100, 128)          1280000   
                                                                 
 bidirectional (Bidirectiona  (None, 128)              98816     
 l)                                                              
                                                     

In [ ]:
selected_column = 'Bengali'  # Change this to 'Bengali' or 'Banglish' to test those!

# ✅ Filter dataset to selected column and label
df_single = df[[selected_column, 'Label']].dropna()
df_single.columns = ['text', 'label']  # Rename for consistency

print("\n✅ Sample Data:\n", df_single.head())

# ---------------------------
# 3. Label Encoding
# ---------------------------

label_encoder = LabelEncoder()
df_single['label'] = label_encoder.fit_transform(df_single['label'])
num_classes = len(label_encoder.classes_)

print("\n✅ Labels Encoded. Classes:", label_encoder.classes_)

# ---------------------------
# 4. Train-Test Split
# ---------------------------

X_train, X_test, y_train, y_test = train_test_split(
    df_single['text'], df_single['label'],
    test_size=0.2,
    stratify=df_single['label'],
    random_state=42
)

# ---------------------------
# 5. Tokenization and Padding
# ---------------------------

max_words = 10000  # Vocabulary size
max_len = 100      # Max sequence length

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# ---------------------------
# 6. One-hot encode labels
# ---------------------------

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

# ---------------------------
# 7. Build CNN/LSTM/GRU Model
# ---------------------------

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    # Change this line to test different models:
    # tf.keras.layers.Conv1D(128, 5, activation='relu'),  # For CNN
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=False)),  # For LSTM
    # tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64, return_sequences=False)),  # For GRU
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# ---------------------------
# 8. Train the Model
# ---------------------------

history = model.fit(
    X_train_pad, y_train_cat,
    epochs=10,
    batch_size=32,
    validation_data=(X_test_pad, y_test_cat)
)

# ---------------------------
# 9. Evaluate Model
# ---------------------------

loss, accuracy = model.evaluate(X_test_pad, y_test_cat)
print(f"\n✅ Test Accuracy on {selected_column}: {accuracy * 100:.2f}%")

# ---------------------------
# 10. Optional: Save Model
# ---------------------------

# model.save(f"text_model_{selected_column}.h5")
# print(f"\n✅ Model saved as text_model_{selected_column}.h5")


✅ Columns: Index(['Label', 'Bengali', 'Banglish', 'English'], dtype='object')

✅ Sample Data:
                                                 text label
0  বাংলাদেশের সীমান্ত লাগোয়া পূর্ব ভারতীয় রাজ্যটি...  fear
1   শেরেবাংলা উচ্চবিদ্যালয়ের মাঠে গভীর নলকূপের পা...  fear
2  তখনি হঠাৎ মিথির দিকে ভালো করে তাকাতে দেখলাম মি...  fear
3  সন্ধ্যায় বাড়ির গৃহিণী ঘরের পাশেই কাজ করছিলেন র...  fear
4  তাহলে কী আমাকে নিয়ে বাজে কোন ফিসফাস চালু হয়ে গ...  fear

✅ Labels Encoded. Classes: ['anger' 'disgust' 'fear' 'joy' 'sadness' 'surprise']
Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 100, 128)          1280000   
                                                                 
 bidirectional_2 (Bidirectio  (None, 128)              98816     
 nal)                                                            
                                                   

In [ ]:
selected_column = 'Banglish'  # For Banglish column
  # Change this to 'Bengali' or 'Banglish' to test those!

# ✅ Filter dataset to selected column and label
df_single = df[[selected_column, 'Label']].dropna()
df_single.columns = ['text', 'label']  # Rename for consistency

print("\n✅ Sample Data:\n", df_single.head())

# ---------------------------
# 3. Label Encoding
# ---------------------------

label_encoder = LabelEncoder()
df_single['label'] = label_encoder.fit_transform(df_single['label'])
num_classes = len(label_encoder.classes_)

print("\n✅ Labels Encoded. Classes:", label_encoder.classes_)

# ---------------------------
# 4. Train-Test Split
# ---------------------------

X_train, X_test, y_train, y_test = train_test_split(
    df_single['text'], df_single['label'],
    test_size=0.2,
    stratify=df_single['label'],
    random_state=42
)

# ---------------------------
# 5. Tokenization and Padding
# ---------------------------

max_words = 10000  # Vocabulary size
max_len = 100      # Max sequence length

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# ---------------------------
# 6. One-hot encode labels
# ---------------------------

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

# ---------------------------
# 7. Build CNN/LSTM/GRU Model
# ---------------------------

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    # Change this line to test different models:
    # tf.keras.layers.Conv1D(128, 5, activation='relu'),  # For CNN
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=False)),  # For LSTM
    # tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64, return_sequences=False)),  # For GRU
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# ---------------------------
# 8. Train the Model
# ---------------------------

history = model.fit(
    X_train_pad, y_train_cat,
    epochs=10,
    batch_size=32,
    validation_data=(X_test_pad, y_test_cat)
)

# ---------------------------
# 9. Evaluate Model
# ---------------------------

loss, accuracy = model.evaluate(X_test_pad, y_test_cat)
print(f"\n✅ Test Accuracy on {selected_column}: {accuracy * 100:.2f}%")

# ---------------------------
# 10. Optional: Save Model
# ---------------------------

# model.save(f"text_model_{selected_column}.h5")
# print(f"\n✅ Model saved as text_model_{selected_column}.h5")


✅ Columns: Index(['Label', 'Bengali', 'Banglish', 'English'], dtype='object')

✅ Sample Data:
                                                 text label
0  bangladesher simanto lagoya purbo vartiy rajjt...  fear
1  sherebangla uccbidjalyer mathe govir nolkuper ...  fear
2  tokhni hothat mithir dike valo kore takate dek...  fear
3  sondhjay barir grihini ghorer pashei kaj korch...  fear
4  tahole ki amake niye baje kon fisfas calu hoye...  fear

✅ Labels Encoded. Classes: ['anger' 'disgust' 'fear' 'joy' 'sadness' 'surprise']
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 100, 128)          1280000   
                                                                 
 bidirectional_1 (Bidirectio  (None, 128)              98816     
 nal)                                                            
                                                   

In [ ]:
#CNN English
selected_column = 'English'  # 👈 Change this to 'Bengali' or 'Banglish' as needed

# ------------------------------------------
# Dataset Pre-processing
# ------------------------------------------
# Clean column names (already done earlier)
df.columns = df.columns.str.replace("'", "").str.strip()

# Check final column names to confirm
print("✅ Columns in dataset:", list(df.columns))

# Prepare 'text' and 'Label'
df['text'] = df[selected_column].fillna('')  # Use only selected column for 'text'
df['label'] = df['Label']  # Assuming 'Label' is your target column

# Encode labels to integers
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])
num_classes = len(label_encoder.classes_)
print(f"✅ Classes found: {label_encoder.classes_}")

# ------------------------------------------
# Data Splitting
# ------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label_encoded'], test_size=0.2, random_state=42, stratify=df['label_encoded']
)

print(f"✅ Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

# ------------------------------------------
# Tokenization and Padding
# ------------------------------------------
max_words = 10000  # Max number of words to consider in tokenizer
max_len = 100     # Max sequence length

tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = tf.keras.preprocessing.sequence.pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = tf.keras.preprocessing.sequence.pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# ------------------------------------------
# CNN Model Architecture
# ------------------------------------------
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    tf.keras.layers.Conv1D(128, 5, activation='relu'),  # CNN layer
    tf.keras.layers.GlobalMaxPooling1D(),  # Global max pooling
    tf.keras.layers.Dense(64, activation='relu'),  # Fully connected
    tf.keras.layers.Dropout(0.3),  # Regularization
    tf.keras.layers.Dense(num_classes, activation='softmax')  # Output layer
])

# ------------------------------------------
# Model Compilation
# ------------------------------------------
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# ------------------------------------------
# Model Summary
# ------------------------------------------
print("\n✅ CNN Model Summary:")
model.summary()

# ------------------------------------------
# Training (with 15 epochs)
# ------------------------------------------
history = model.fit(
    X_train_pad, y_train,
    epochs=15,  # 🔥 Running for 15 epochs as requested
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

# ------------------------------------------
# Evaluation
# ------------------------------------------
loss, accuracy = model.evaluate(X_test_pad, y_test, verbose=1)
print(f"\n✅ Test Accuracy on {selected_column} data: {accuracy * 100:.2f}%")


✅ Columns in dataset: ['Label', 'Bengali', 'Banglish', 'English', 'text', 'label', 'label_encoded']
✅ Classes found: ['anger' 'disgust' 'fear' 'joy' 'sadness' 'surprise']
✅ Training samples: 64078, Testing samples: 16020

✅ CNN Model Summary:
Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_6 (Embedding)     (None, 100, 128)          1280000   
                                                                 
 conv1d_3 (Conv1D)           (None, 96, 128)           82048     
                                                                 
 global_max_pooling1d_1 (Glo  (None, 128)              0         
 balMaxPooling1D)                                                
                                                                 
 dense_12 (Dense)            (None, 64)                8256      
                                                                 
 dropout_

In [ ]:
# Hybrid
# 🚀 1. Import Libraries
# -------------------------
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import re

# -------------------------
# 🚀 2. Dataset Cleaning
# -------------------------
# Assuming 'df' is already loaded and contains proper columns
# You mentioned column names: "'Label'", "'Bengali'", "'Banglish'", "'English'"
df.columns = [col.strip().replace("'", "") for col in df.columns]  # Clean column names

# Select only 'English' and 'Label'
df = df[['English', 'Label']].dropna()  # Drop rows where any of these columns are null

# Clean text function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # Keep only letters and spaces
    text = re.sub(r"\s+", " ", text).strip()  # Remove extra spaces
    return text

# Apply text cleaning
df['English'] = df['English'].apply(clean_text)

# -------------------------
# 🚀 3. Encode Labels
# -------------------------
label_encoder = LabelEncoder()
df['Label'] = label_encoder.fit_transform(df['Label'])

num_classes = len(label_encoder.classes_)  # Number of unique classes

# -------------------------
# 🚀 4. Prepare Tokenizer and Padding
# -------------------------
max_words = 20000  # Vocabulary size
max_len = 200      # Max sequence length

tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(df['English'])

# Convert texts to sequences
sequences = tokenizer.texts_to_sequences(df['English'])
X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')
y = df['Label'].values

# -------------------------
# 🚀 5. Train-Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# -------------------------
# 🚀 6. Build Hybrid CNN + LSTM Model
# -------------------------
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    Conv1D(128, 5, activation='relu'),
    MaxPooling1D(pool_size=2),
    LSTM(64, return_sequences=False),  # LSTM after CNN
    Dense(64, activation='relu'),
    Dropout(0.4),  # Regularization
    Dense(num_classes, activation='softmax')  # Output layer for multi-class classification
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# -------------------------
# 🚀 7. Callbacks for Training
# -------------------------
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, verbose=1)

# -------------------------
# 🚀 8. Model Training
# -------------------------
history = model.fit(
    X_train, y_train,
    epochs=15,  # You requested 15 epochs
    batch_size=64,  # Can adjust to 32/128 based on performance
    validation_split=0.1,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

# -------------------------
# 🚀 9. Evaluation
# -------------------------
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"\n✅ Final Test Accuracy: {accuracy * 100:.2f}%")

# -------------------------
# 🚀 10. Optional: Label Mapping Back
# -------------------------
print("\nLabel Mapping:", dict(zip(label_encoder.classes_, range(len(label_encoder.classes_)))))


Epoch 1/15
902/902 [==============================] - 10s 10ms/step - loss: 1.7580 - accuracy: 0.2155 - val_loss: 1.7565 - val_accuracy: 0.2208 - lr: 0.0010
Epoch 2/15
902/902 [==============================] - 9s 10ms/step - loss: 1.7550 - accuracy: 0.2214 - val_loss: 1.7556 - val_accuracy: 0.2213 - lr: 0.0010
Epoch 3/15
902/902 [==============================] - 9s 10ms/step - loss: 1.7538 - accuracy: 0.2230 - val_loss: 1.7552 - val_accuracy: 0.2216 - lr: 0.0010
Epoch 4/15
902/902 [==============================] - 9s 10ms/step - loss: 1.7529 - accuracy: 0.2240 - val_loss: 1.7558 - val_accuracy: 0.2219 - lr: 0.0010
Epoch 5/15
902/902 [==============================] - 9s 10ms/step - loss: 1.6750 - accuracy: 0.2847 - val_loss: 1.5059 - val_accuracy: 0.3766 - lr: 0.0010
Epoch 6/15
902/902 [==============================] - 9s 10ms/step - loss: 1.4320 - accuracy: 0.4185 - val_loss: 1.3741 - val_accuracy: 0.4440 - lr: 0.0010
Epoch 7/15
902/902 [==============================] - 9s 10ms/s

In [ ]:
# CNN-------------------------

df['English'] = df['English'].apply(clean_text)

print("✅ Dataset cleaned and ready for processing")

# -------------------------
# 🚀 3. Encode Labels
# -------------------------
label_encoder = LabelEncoder()
df['Label'] = label_encoder.fit_transform(df['Label'])

num_classes = len(label_encoder.classes_)  # Number of unique classes
print(f"✅ Number of classes: {num_classes}")

# -------------------------
# 🚀 4. Prepare Tokenizer and Padding
# -------------------------
max_words = 20000  # Vocabulary size
max_len = 200      # Max sequence length

tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(df['English'])

# Convert texts to sequences
sequences = tokenizer.texts_to_sequences(df['English'])
X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')
y = df['Label'].values

print(f"✅ Data tokenized and padded. Shape: {X.shape}")

# -------------------------
# 🚀 5. Train-Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("✅ Data split into train and test sets")

# -------------------------
# 🚀 6. Build CNN Model (only CNN)
# -------------------------
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    Conv1D(128, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Conv1D(64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),  # Regularization
    Dense(num_classes, activation='softmax')  # Output layer for multi-class classification
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("✅ CNN model built successfully")
model.summary()

# -------------------------
# 🚀 7. Callbacks for Training
# -------------------------
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, verbose=1)

# -------------------------
# 🚀 8. Model Training
# -------------------------
history = model.fit(
    X_train, y_train,
    epochs=15,  # You want 15 epochs
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

# -------------------------
# 🚀 9. Evaluation
# -------------------------
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"\n✅ Final Test Accuracy: {accuracy * 100:.2f}%")

# -------------------------
# 🚀 10. Optional: Label Mapping Back
# -------------------------
print("\nLabel Mapping:", dict(zip(label_encoder.classes_, range(len(label_encoder.classes_)))))


✅ Dataset cleaned and ready for processing
✅ Number of classes: 6
✅ Data tokenized and padded. Shape: (80098, 200)
✅ Data split into train and test sets
✅ CNN model built successfully
Model: "sequential_8"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_8 (Embedding)     (None, 200, 128)          2560000   
                                                                 
 conv1d_5 (Conv1D)           (None, 196, 128)          82048     
                                                                 
 max_pooling1d_1 (MaxPooling  (None, 98, 128)          0         
 1D)                                                             
                                                                 
 conv1d_6 (Conv1D)           (None, 94, 64)            41024     
                                                                 
 max_pooling1d_2 (MaxPooling  (None, 47, 64)           0         
 1

In [ ]:
df.columns = df.columns.str.strip()  # Strip extra spaces
print("Initial Column Names:", repr(df.columns))

# Rename 'English' to 'Banglish' if exists
if 'English' in df.columns:
    df = df.rename(columns={'English': 'Banglish'})
else:
    print("Column 'English' not found!")

# Verify column names after renaming
print("Renamed Column Names:", repr(df.columns))

# Select only 'Banglish' and 'Label' columns and drop rows with missing values
df = df[['Banglish', 'Label']].dropna()  # Drop rows where any of these columns are null
print("Data after selection:", df.head())

# -------------------------
# 🚀 3. Encode Labels
# -------------------------
label_encoder = LabelEncoder()
df['Label'] = label_encoder.fit_transform(df['Label'])

num_classes = len(label_encoder.classes_)  # Number of unique classes
print(f"✅ Number of classes: {num_classes}")

# -------------------------
# 🚀 4. Prepare Tokenizer and Padding
# -------------------------
max_words = 20000  # Vocabulary size
max_len = 200      # Max sequence length

tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(df['Banglish'])

# Convert texts to sequences
sequences = tokenizer.texts_to_sequences(df['Banglish'])
X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')
y = df['Label'].values

print(f"✅ Data tokenized and padded. Shape: {X.shape}")

# -------------------------
# 🚀 5. Train-Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("✅ Data split into train and test sets")

# -------------------------
# 🚀 6. Build CNN Model (only CNN)
# -------------------------
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    Conv1D(128, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Conv1D(64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),  # Regularization
    Dense(num_classes, activation='softmax')  # Output layer for multi-class classification
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("✅ CNN model built successfully")
model.summary()

# -------------------------
# 🚀 7. Callbacks for Training
# -------------------------
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, verbose=1)

# -------------------------
# 🚀 8. Model Training
# -------------------------
history = model.fit(
    X_train, y_train,
    epochs=15,  # You want 15 epochs
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

# -------------------------
# 🚀 9. Evaluation
# -------------------------
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"\n✅ Final Test Accuracy: {accuracy * 100:.2f}%")

# -------------------------
# 🚀 10. Optional: Label Mapping Back
# -------------------------
print("\nLabel Mapping:", dict(zip(label_encoder.classes_, range(len(label_encoder.classes_)))))


Initial Column Names: Index(['English', 'Label'], dtype='object')
Renamed Column Names: Index(['Banglish', 'Label'], dtype='object')
Data after selection:                                             Banglish  Label
0  muslims are under severe stress in the eastern...      2
1  on thursday gas continued to escape at a high ...      2
2  then suddenly i looked closely at mithi and sa...      2
3  in the evening the housewife was working next ...      2
4  so whats wrong with me has started a whisper l...      2
✅ Number of classes: 6
✅ Data tokenized and padded. Shape: (80098, 200)
✅ Data split into train and test sets
✅ CNN model built successfully
Model: "sequential_9"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_9 (Embedding)     (None, 200, 128)          2560000   
                                                                 
 conv1d_7 (Conv1D)           (None, 196, 128)          820

In [ ]:
df.columns = df.columns.str.strip()  # Strip extra spaces
print("Initial Column Names:", repr(df.columns))

# Rename 'English' to 'Banglish' if exists
if 'English' in df.columns:
    df = df.rename(columns={'English': 'Banglish'})
else:
    print("Column 'English' not found!")

# Verify column names after renaming
print("Renamed Column Names:", repr(df.columns))

# Select only 'Banglish' and 'Label' columns and drop rows with missing values
df = df[['Banglish', 'Label']].dropna()  # Drop rows where any of these columns are null
print("Data after selection:", df.head())

# -------------------------
# 🚀 3. Bengali Text Cleaning
# -------------------------
def clean_bengali_text(text):
    # Remove non-Bengali characters and keep only Bengali text and spaces
    text = re.sub(r"[^a-zA-Z০-৯\u0980-\u09FF\s]", "", text)  # Keep Bengali letters, digits, and spaces
    text = re.sub(r"\s+", " ", text).strip()  # Remove extra spaces
    return text

# Apply cleaning function to the 'Banglish' column
df['Banglish'] = df['Banglish'].apply(clean_bengali_text)

# Check the cleaned text
print("Cleaned data:", df.head())

# -------------------------
# 🚀 4. Encode Labels
# -------------------------
label_encoder = LabelEncoder()
df['Label'] = label_encoder.fit_transform(df['Label'])

num_classes = len(label_encoder.classes_)  # Number of unique classes
print(f"✅ Number of classes: {num_classes}")

# -------------------------
# 🚀 5. Prepare Tokenizer and Padding
# -------------------------
max_words = 20000  # Vocabulary size
max_len = 200      # Max sequence length

tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(df['Banglish'])

# Convert texts to sequences
sequences = tokenizer.texts_to_sequences(df['Banglish'])
X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')
y = df['Label'].values

print(f"✅ Data tokenized and padded. Shape: {X.shape}")

# -------------------------
# 🚀 6. Train-Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("✅ Data split into train and test sets")

# -------------------------
# 🚀 7. Build CNN Model (only CNN)
# -------------------------
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    Conv1D(128, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Conv1D(64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),  # Regularization
    Dense(num_classes, activation='softmax')  # Output layer for multi-class classification
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("✅ CNN model built successfully")
model.summary()

# -------------------------
# 🚀 8. Callbacks for Training
# -------------------------
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, verbose=1)

# -------------------------
# 🚀 9. Model Training
# -------------------------
history = model.fit(
    X_train, y_train,
    epochs=15,  # You want 15 epochs
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

# -------------------------
# 🚀 10. Evaluation
# -------------------------
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"\n✅ Final Test Accuracy: {accuracy * 100:.2f}%")

# -------------------------
# 🚀 11. Optional: Label Mapping Back
# -------------------------
print("\nLabel Mapping:", dict(zip(label_encoder.classes_, range(len(label_encoder.classes_)))))


Initial Column Names: Index(['Banglish', 'Label'], dtype='object')
Column 'English' not found!
Renamed Column Names: Index(['Banglish', 'Label'], dtype='object')
Data after selection:                                             Banglish  Label
0  muslims are under severe stress in the eastern...      2
1  on thursday gas continued to escape at a high ...      2
2  then suddenly i looked closely at mithi and sa...      2
3  in the evening the housewife was working next ...      2
4  so whats wrong with me has started a whisper l...      2
Cleaned data:                                             Banglish  Label
0  muslims are under severe stress in the eastern...      2
1  on thursday gas continued to escape at a high ...      2
2  then suddenly i looked closely at mithi and sa...      2
3  in the evening the housewife was working next ...      2
4  so whats wrong with me has started a whisper l...      2
✅ Number of classes: 6
✅ Data tokenized and padded. Shape: (80098, 200)
✅ Data spl

In [ ]:
# Load the Excel dataset
file_path = "Bengali_Banglish_Dataset12.xlsx"  # Make sure this file is in the same directory
df = pd.read_excel(file_path)


In [ ]:
train_texts = df['Banglish'].astype(str).values.tolist()  # Convert to list of strings
train_labels = df['Label'].values

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(train_texts, train_labels, test_size=0.2, random_state=42)

# -------------------------------
# Preprocessing function
# -------------------------------
def preprocess_text(text):
    # Lowercase the text
    text = text.lower()

    # Remove numbers, special characters, and extra spaces
    text = re.sub(r'[^a-z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Preprocess both training and test data
X_train = [preprocess_text(text) for text in X_train]
X_test = [preprocess_text(text) for text in X_test]

# -------------------------------
# Tokenize the text data using Keras Tokenizer
# -------------------------------
max_words = 10000  # You can adjust this based on the dataset
max_len = 128  # Maximum length of the sequences

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)  # Fit the tokenizer on the training texts

# Convert texts to sequences (lists of integers)
train_sequences = tokenizer.texts_to_sequences(X_train)
test_sequences = tokenizer.texts_to_sequences(X_test)

# -------------------------------
# Pad the sequences to ensure they have the same length
# -------------------------------
train_padded = pad_sequences(train_sequences, maxlen=max_len, padding='post', truncating='post')
test_padded = pad_sequences(test_sequences, maxlen=max_len, padding='post', truncating='post')

# -------------------------------
# Label encoding (convert labels to numerical values)
# -------------------------------
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# -------------------------------
# Create the GRU model
# -------------------------------
model = Sequential([
    Embedding(input_dim=max_words, output_dim=100, input_length=max_len),
    GRU(64, return_sequences=True),
    Dropout(0.2),
    GRU(64),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(6, activation='softmax')  # Assuming 6 classes for classification
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# -------------------------------
# Train the model
# -------------------------------
history = model.fit(
    train_padded, y_train_encoded,
    validation_data=(test_padded, y_test_encoded),
    epochs=15,  # Set epochs to 15
    batch_size=64,  # Adjusted batch size for better GPU utilization
    verbose=1,
    use_multiprocessing=True,  # Use multiprocessing for faster data loading
    workers=4  # Set workers for parallel data processing
)

# -------------------------------
# Evaluate the model
# -------------------------------
loss, accuracy = model.evaluate(test_padded, y_test_encoded, verbose=1)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")


Epoch 1/15
1002/1002 [==============================] - 18s 17ms/step - loss: 1.7569 - accuracy: 0.2181 - val_loss: 1.7528 - val_accuracy: 0.2267
Epoch 2/15
1002/1002 [==============================] - 19s 19ms/step - loss: 1.7555 - accuracy: 0.2208 - val_loss: 1.7524 - val_accuracy: 0.2267
Epoch 3/15
1002/1002 [==============================] - 30s 30ms/step - loss: 1.7549 - accuracy: 0.2210 - val_loss: 1.7521 - val_accuracy: 0.2275
Epoch 4/15
1002/1002 [==============================] - 28s 28ms/step - loss: 1.7537 - accuracy: 0.2218 - val_loss: 1.7520 - val_accuracy: 0.2275
Epoch 5/15
1002/1002 [==============================] - 24s 24ms/step - loss: 1.7525 - accuracy: 0.2217 - val_loss: 1.7523 - val_accuracy: 0.2273
Epoch 6/15
1002/1002 [==============================] - 26s 26ms/step - loss: 1.7519 - accuracy: 0.2228 - val_loss: 1.7529 - val_accuracy: 0.2272
Epoch 7/15
1002/1002 [==============================] - 16s 16ms/step - loss: 1.7517 - accuracy: 0.2230 - val_loss: 1.7531 -